In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

RAW = Path("../data/raw/dataset")

t = {f.stem: pd.read_csv(f) for f in RAW.glob("*.csv")}

for nom, df in t.items():
    print(f"{nom:25s} {df.shape}")

assures                   (4459, 9)
contrats                  (1500, 12)
factures                  (84593, 24)
garanties                 (27, 11)
grille_tarifaire          (3242, 3)
lignes_facture            (123270, 9)
ordonnances               (47804, 8)
ordonnance_lignes         (82175, 6)
prestataires              (251, 20)
referentiel_actes         (58, 12)
referentiel_medicaments   (28, 6)


In [3]:
for nom in ["factures", "ordonnances", "assures", "prestataires", "contrats"]:
    print("\n==", nom)
    manquants = (t[nom].isna().mean() * 100).round(1)
    print(manquants[manquants > 0])


== factures
ordonnance_id           44.6
attestation_pec_id      79.9
entente_prealable_id    99.6
motif_rejet             97.6
date_traitement          3.3
dtype: float64

== ordonnances
diagnostic                0.4
renseignement_clinique    0.4
dtype: float64

== assures
date_radiation    92.6
dtype: float64

== prestataires
specialite             83.3
numero_onm             54.2
date_fin_convention    96.4
dtype: float64

== contrats
date_fin    92.7
dtype: float64


In [4]:
for nom, df in t.items():
    print("=" * 60)
    print(nom, df.shape)
    print("=" * 60)
    df.info()
    print("\n-- describe --")
    print(df.describe().round(1))
    print("\n-- valeurs manquantes --")
    print(df.isna().sum()[df.isna().sum() > 0])
    print("\n-- lignes dupliquées :", df.duplicated().sum(), "\n")

assures (4459, 9)
<class 'pandas.DataFrame'>
RangeIndex: 4459 entries, 0 to 4458
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   assure_id            4459 non-null   str  
 1   contrat_id           4459 non-null   str  
 2   rang_beneficiaire    4459 non-null   str  
 3   assure_principal_id  4459 non-null   str  
 4   sexe                 4459 non-null   str  
 5   annee_naissance      4459 non-null   int64
 6   ville                4459 non-null   str  
 7   date_adhesion        4459 non-null   str  
 8   date_radiation       329 non-null    str  
dtypes: int64(1), str(8)
memory usage: 621.9 KB

-- describe --
       annee_naissance
count           4459.0
mean            1997.9
std               19.4
min             1955.0
25%             1981.0
50%             2001.0
75%             2016.0
max             2024.0

-- valeurs manquantes --
date_radiation    4130
dtype: int64

-- lignes dupliquées : 0 

In [5]:
# 1. Chaque clé primaire est-elle unique ?
cles_primaires = {
    "assures": "assure_id",
    "contrats": "contrat_id",
    "factures": "facture_id",
    "lignes_facture": "ligne_id",
    "ordonnances": "ordonnance_id",
    "prestataires": "prestataire_id",
    "referentiel_actes": "code_acte",
    "referentiel_medicaments": "code_medicament",
}
print("--- Unicité des clés primaires ---")
for table, cle in cles_primaires.items():
    print(f"{table:25s} {cle:18s} unique : {t[table][cle].is_unique}")

# 2. Chaque clé étrangère pointe-t-elle vers une ligne existante ?
codes_connus = set(t["referentiel_actes"].code_acte) | set(t["referentiel_medicaments"].code_medicament)

relations = [
    ("factures", "prestataire_id", set(t["prestataires"].prestataire_id)),
    ("factures", "assure_id", set(t["assures"].assure_id)),
    ("factures", "contrat_id", set(t["contrats"].contrat_id)),
    ("factures", "ordonnance_id", set(t["ordonnances"].ordonnance_id)),
    ("lignes_facture", "facture_id", set(t["factures"].facture_id)),
    ("lignes_facture", "code", codes_connus),
    ("ordonnances", "assure_id", set(t["assures"].assure_id)),
    ("ordonnances", "prescripteur_id", set(t["prestataires"].prestataire_id)),
    ("ordonnance_lignes", "ordonnance_id", set(t["ordonnances"].ordonnance_id)),
    ("ordonnance_lignes", "code", codes_connus),
    ("assures", "contrat_id", set(t["contrats"].contrat_id)),
    ("assures", "assure_principal_id", set(t["assures"].assure_id)),
    ("grille_tarifaire", "prestataire_id", set(t["prestataires"].prestataire_id)),
    ("grille_tarifaire", "code", codes_connus),
    ("contrats", "formule", set(t["garanties"].formule)),
]

print("\n--- Valeurs orphelines (absentes de la table de référence) ---")
for table, col, ref in relations:
    valeurs = t[table][col].dropna()
    orphelins = (~valeurs.isin(ref)).sum()
    print(f"{table:20s}.{col:20s} -> {orphelins} orphelin(s)")

--- Unicité des clés primaires ---
assures                   assure_id          unique : True
contrats                  contrat_id         unique : True
factures                  facture_id         unique : True
lignes_facture            ligne_id           unique : True
ordonnances               ordonnance_id      unique : True
prestataires              prestataire_id     unique : True
referentiel_actes         code_acte          unique : True
referentiel_medicaments   code_medicament    unique : True

--- Valeurs orphelines (absentes de la table de référence) ---
factures            .prestataire_id       -> 0 orphelin(s)
factures            .assure_id            -> 0 orphelin(s)
factures            .contrat_id           -> 0 orphelin(s)
factures            .ordonnance_id        -> 0 orphelin(s)
lignes_facture      .facture_id           -> 0 orphelin(s)
lignes_facture      .code                 -> 0 orphelin(s)
ordonnances         .assure_id            -> 0 orphelin(s)
ordonnances     